In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42  
TRAIN_PATH = "../../../src/data/classificationdata/H2_classification_train.csv"
TEST_PATH = "../../../src/data/classificationdata/H2_classification_test.csv"
TARGET = "IsCanceled"

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

print("train:", train.shape)
print("test :", test.shape)

train: (43332, 54)
test : (12343, 54)


In [3]:
# #Cell 3: basic checks
# The first line checks both files have the same columns in the same order. If not, the model will break.
# isna().sum().sum() counts empty cells. We want 0.
# duplicated().sum() counts repeated rows.
# value_counts(normalize=True) shows the share of 0s and 1s. About 42% of the original data is canceled. If the rates differ a lot between files, something is wrong.
print("Same columns in both files:", list(train.columns) == list(test.columns))
print("Missing values  train:", train.isna().sum().sum(), " test:", test.isna().sum().sum())
print("Duplicate rows  train:", train.duplicated().sum(), " test:", test.duplicated().sum())

print("\nCancel rate (1 = canceled)")
print("train:", train[TARGET].value_counts(normalize=True).round(3).to_dict())
print("test :", test[TARGET].value_counts(normalize=True).round(3).to_dict())

Same columns in both files: True
Missing values  train: 0  test: 0
Duplicate rows  train: 653  test: 367

Cancel rate (1 = canceled)
train: {0: 0.698, 1: 0.302}
test : {0: 0.675, 1: 0.325}


In [4]:
overlap = train.merge(test, how="inner")   # rows that appear in both files
print("Rows that appear in both train and test:", len(overlap))

Rows that appear in both train and test: 89


In [5]:
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]

X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
print(list(X_train.columns))

(43332, 53) (43332,)
(12343, 53) (12343,)
['LeadTime', 'TotalOfSpecialRequests', 'month_1', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12', 'TotalNights', 'WeekendOrNo', 'IsZeroNights', 'total_guests', 'Meal_BB', 'Meal_FB', 'Meal_HB', 'Meal_SC', 'Country_AUT', 'Country_BEL', 'Country_BRA', 'Country_CHE', 'Country_CHN', 'Country_DEU', 'Country_ESP', 'Country_FRA', 'Country_GBR', 'Country_IRL', 'Country_ITA', 'Country_NLD', 'Country_PRT', 'Country_USA', 'Country_infrequent_sklearn', 'MarketSegment_Aviation', 'MarketSegment_Complementary', 'MarketSegment_Corporate', 'MarketSegment_Direct', 'MarketSegment_Groups', 'MarketSegment_Offline TA/TO', 'MarketSegment_Online TA', 'has_prev_not_canceled', 'DepositType_No Deposit', 'DepositType_Non Refund', 'DepositType_Refundable', 'Agent_te', 'has_company', 'CustomerType_Transient_Party', 'CustomerType_Contract', 'CustomerType_Group']


In [6]:
#Which columns are NOT 0/1? These show whether the numbers are scaled already
#Models like KNN, SVM and Logistic Regression need scaled numbers. 
#If LeadTime goes up to hundreds, we scale it inside the pipeline. If it is already small, the team scaled it
not_binary = [c for c in X_train.columns if X_train[c].nunique() > 2]
print(X_train[not_binary].describe().round(2).T[["min", "mean", "max"]])

                        min   mean    max
LeadTime                0.0  78.49  629.0
TotalOfSpecialRequests  0.0   0.71    5.0
TotalNights             0.0   3.14   48.0
total_guests            1.0   2.02   12.0
Agent_te                0.0   0.30    1.0


In [7]:
#shows how strongly each column moves with IsCanceled. A value close to 1 (above about 0.5) would be a warning of leakage
corr = X_train.corrwith(y_train).abs().sort_values(ascending=False)
print(corr.head(10).round(3))

Agent_te                        0.274
DepositType_No Deposit          0.200
DepositType_Non Refund          0.200
LeadTime                        0.194
MarketSegment_Online TA         0.180
TotalOfSpecialRequests          0.165
CustomerType_Transient_Party    0.121
TotalNights                     0.117
MarketSegment_Offline TA/TO     0.109
MarketSegment_Direct            0.104
dtype: float64


In [8]:
# This trains a tiny tree on one column at a time and measures how well that column alone predicts cancellation.
# ROC-AUC runs from 0.5 (random guessing) to 1.0 (perfect).
# A single column above about 0.9 would be suspicious, because real booking details rarely predict that well alone. 
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

scores = {}
for col in X_train.columns:
    tree = DecisionTreeClassifier(max_depth=3, random_state=SEED)
    scores[col] = cross_val_score(tree, X_train[[col]], y_train,
                                  cv=3, scoring="roc_auc").mean()

print(pd.Series(scores).sort_values(ascending=False).head(10).round(3))

Agent_te                        0.651
LeadTime                        0.629
TotalOfSpecialRequests          0.602
MarketSegment_Online TA         0.594
TotalNights                     0.555
total_guests                    0.548
CustomerType_Transient_Party    0.545
Country_PRT                     0.544
MarketSegment_Offline TA/TO     0.541
MarketSegment_Direct            0.534
dtype: float64


In [9]:
for col in ["DepositType_No Deposit", "DepositType_Non Refund", "DepositType_Refundable"]:
    rate = y_train[X_train[col] == 1].mean()
    count = int(X_train[col].sum())
    print(f"{col}: {count} bookings, cancel rate {rate:.2f}")

DepositType_No Deposit: 42532 bookings, cancel rate 0.29
DepositType_Non Refund: 787 bookings, cancel rate 0.98
DepositType_Refundable: 13 bookings, cancel rate 0.69


In [10]:
# Compare all models with 5-fold cross-validation on the train file only. 
# The train data is cut into 5 parts, and each part takes a turn as the "exam" while the model learns from the other 4.
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold

cv  = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)  # 10 fits # This gives 10 scores per model instead of 5. Runs take twice as long
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)         # 5 fits

In [11]:
# a helper function that scores any model
import time
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.metrics import make_scorer, recall_score

def weighted_hm(y_true, y_pred, w_recall=2, w_spec=1):
    recall = recall_score(y_true, y_pred, pos_label=1)
    spec = recall_score(y_true, y_pred, pos_label=0) #Specificity=TN/TN+FP
    if recall == 0 or spec == 0:
        return 0.0
    return (w_recall + w_spec) / (w_recall / recall + w_spec / spec)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "hm_1_1": make_scorer(weighted_hm, w_recall=1, w_spec=1),#Recall and Specificity have equal weights
    "hm_2_1": make_scorer(weighted_hm, w_recall=2, w_spec=1),#Recall gets twice the weight
    "hm_3_1": make_scorer(weighted_hm, w_recall=3, w_spec=1)#Recall gets three times the weight
}

def run_cv(name, model, scale=False, cv_obj=None):
    """Cross-validate one model on the train data and return a result row."""
    cv_obj = cv_obj if cv_obj is not None else cv   # default: the repeated CV
    if scale:
        model = Pipeline([("scaler", StandardScaler()), ("model", model)])

    start = time.time()
    res = cross_validate(model, X_train, y_train, cv=cv_obj, scoring=scoring, n_jobs=-1)
    total_time = time.time() - start

    row = {"model": name}
    for metric in scoring:
        row[f"cv_{metric}"] = res[f"test_{metric}"].mean()
    row["cv_f1_std"] = res["test_f1"].std()
    row["time_sec"] = total_time
    row["cv_scheme"] = f"{len(res['test_f1'])} fits"   # 10 = repeated, 5 = single
    return row

In [12]:
#test helper function for one model
from sklearn.linear_model import LogisticRegression

row = run_cv("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED), scale=True)
pd.DataFrame([row]).round(4)

,model,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_hm_1_1,cv_hm_2_1,cv_hm_3_1,cv_f1_std,time_sec,cv_scheme
0,LogisticRegression,0.7678,0.6641,0.4691,0.5497,0.8107,0.6569,0.6159,0.5577,0.5326,0.0097,3.7013,10 fits


In [ ]:
#Train all 8 candidate models with default settings and compare them by cross-validation
#Linear Regression (baseline). 
# K-Nearest Neighbors Regressor. 
# Decision Tree Regressor. 
# Random Forest Regressor. 
# XGBoost Regressor or another approved gradient-boosted tree implementation. 
# Multilayer Perceptron / Neural Network Regressor.
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# name -> (model, needs scaling?)
models = {
    "GaussianNB":         (GaussianNB(), False),
    "LogisticRegression": (LogisticRegression(max_iter=1000, random_state=SEED), True),
    "KNN":                (KNeighborsClassifier(), True),
    "DecisionTree":       (DecisionTreeClassifier(random_state=SEED), False),
    "RandomForest":       (RandomForestClassifier(random_state=SEED, n_jobs=-1), False),
    "XGBoost":            (XGBClassifier(random_state=SEED, eval_metric="logloss"), False),
    "MLP":                (MLPClassifier(max_iter=300, random_state=SEED), True),
}

rows = []
for name, (model, scale) in models.items():
    print("running", name, "...", flush=True)
    # if name == "MLP":
    #     rows.append(
    #         run_cv(name, model, scale=scale, cv_obj=cv5)
    #     )
    # else:
    rows.append(
            run_cv(name, model, scale=scale)
        )

running GaussianNB ...
running LogisticRegression ...
running KNN ...
running DecisionTree ...
running RandomForest ...


In [ ]:
#SVM
from sklearn.svm import SVC

# SVM is slow on 43,000 rows, so it gets its own cell.
rows.append(run_cv("SVM_rbf", SVC(kernel="rbf", random_state=SEED), scale=True))

In [ ]:
from pathlib import Path

baseline = pd.DataFrame(rows).sort_values("cv_pr_auc", ascending=False).round(4)
display(baseline)

Path("../../../outputs").mkdir(parents=True, exist_ok=True)
baseline.to_csv("../../../outputs/classification_Cross-validation_Results.csv", index=False)

,model,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_hm_1_1,cv_hm_2_1,cv_hm_3_1,cv_f1_std,time_sec,cv_scheme
5,XGBoost,0.8001,0.7126,0.5725,0.6278,0.8640,0.7364,0.6920,0.6464,0.6261,0.0744,8.2197,10 fits
4,RandomForest,0.8010,0.7059,0.5864,0.6403,0.8586,0.7255,0.7079,0.6621,0.6414,0.0109,19.3140,10 fits
6,MLP,0.7840,0.6579,0.5950,0.6246,0.8429,0.7032,0.7050,0.6640,0.6453,0.0111,202.1321,10 fits
7,SVM_rbf,0.7904,0.7096,0.5193,0.5996,0.8362,0.6991,0.6606,0.6057,0.5815,0.0073,398.7072,10 fits
1,LogisticRegression,0.7678,0.6641,0.4691,0.5497,0.8107,0.6569,0.6159,0.5577,0.5326,0.0097,2.9326,10 fits
2,KNN,0.7524,0.6014,0.5357,0.5666,0.7791,0.5715,0.6559,0.6102,0.5897,0.0106,32.6761,10 fits
0,GaussianNB,0.6732,0.4718,0.6518,0.5464,0.7227,0.5475,0.6649,0.6601,0.6578,0.0064,18.4137,10 fits
3,DecisionTree,0.7438,0.5770,0.5742,0.5750,0.6964,0.4608,0.6737,0.6368,0.6199,0.0145,4.5934,10 fits


In [ ]:
#XGBboost problem with f1_std having a very high value.
xgb_res = cross_validate(XGBClassifier(random_state=SEED, eval_metric="logloss"),
                         X_train, y_train, cv=cv,
                         scoring={"f1": "f1", "recall": "recall", "roc_auc": "roc_auc"}, n_jobs=-1)
print(pd.DataFrame({k: v for k, v in xgb_res.items() if k.startswith("test_")}).round(3))

   test_f1  test_recall  test_roc_auc
0    0.649        0.589         0.873
1    0.656        0.606         0.867
2    0.655        0.619         0.868
3    0.405        0.277         0.829
4    0.652        0.607         0.866
5    0.660        0.611         0.876
6    0.658        0.615         0.870
7    0.653        0.606         0.865
8    0.648        0.603         0.865
9    0.642        0.592         0.862


In [ ]:
folds = list(cv.split(X_train, y_train))   # same cuts as before
tr_idx, va_idx = folds[3]                   # the bad fold

def fold_report(model, cols, label):
    model.fit(X_train.iloc[tr_idx][cols], y_train.iloc[tr_idx])
    pred = model.predict(X_train.iloc[va_idx][cols])
    prob = model.predict_proba(X_train.iloc[va_idx][cols])[:, 1]
    print(label,
          "| recall:", round(recall_score(y_train.iloc[va_idx], pred), 3),
          "| ROC-AUC:", round(roc_auc_score(y_train.iloc[va_idx], prob), 3))
    return pred

from sklearn.metrics import roc_auc_score
all_cols = X_train.columns.tolist()
no_agent = [c for c in all_cols if c != "Agent_te"]

pred = fold_report(XGBClassifier(random_state=SEED, eval_metric="logloss"), all_cols, "XGB, all columns  ")
fold_report(XGBClassifier(random_state=SEED, eval_metric="logloss"), no_agent, "XGB, no Agent_te   ")
fold_report(RandomForestClassifier(random_state=SEED, n_jobs=-1), all_cols, "RF,  all columns  ")

XGB, all columns   | recall: 0.277 | ROC-AUC: 0.829
XGB, no Agent_te    | recall: 0.544 | ROC-AUC: 0.84
RF,  all columns   | recall: 0.527 | ROC-AUC: 0.855


array([0, 1, 0, ..., 0, 0, 0], shape=(8666,))

In [ ]:
va = X_train.iloc[va_idx].copy()
va["y"] = y_train.iloc[va_idx].values
va["pred"] = pred

missed = va[(va["y"] == 1) & (va["pred"] == 0)]     # canceled, but predicted "stays"
caught = va[(va["y"] == 1) & (va["pred"] == 1)]     # canceled, and predicted correctly

compare = pd.DataFrame({"missed": missed.mean(), "caught": caught.mean()})
compare["gap"] = (compare["missed"] - compare["caught"]).abs()
display(compare.sort_values("gap", ascending=False).head(8).round(3))

,missed,caught,gap
LeadTime,88.793,145.912,57.119
pred,0.000,1.000,1.000
TotalNights,3.269,4.105,0.836
Country_PRT,0.148,0.688,0.540
TotalOfSpecialRequests,0.649,0.229,0.420
MarketSegment_Online TA,0.861,0.610,0.251
DepositType_No Deposit,1.000,0.797,0.203
DepositType_Non Refund,0.000,0.203,0.203


In [ ]:
def quick_cv(model, cols):
    res = cross_validate(model, X_train[cols], y_train, cv=cv, n_jobs=-1,
                         scoring={"f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision"})
    out = {k.replace("test_", ""): round(v.mean(), 4) for k, v in res.items() if k.startswith("test_")}
    out["f1_std"] = round(res["test_f1"].std(), 4)
    return out

for name in ["RandomForest", "XGBoost"]:
    model = models[name][0]
    print(name, "| all columns :", quick_cv(model, all_cols))
    print(name, "| no Agent_te :", quick_cv(model, no_agent))

RandomForest | all columns : {'f1': np.float64(0.6403), 'roc_auc': np.float64(0.8586), 'pr_auc': np.float64(0.7255), 'f1_std': np.float64(0.0109)}
RandomForest | no Agent_te : {'f1': np.float64(0.5924), 'roc_auc': np.float64(0.8227), 'pr_auc': np.float64(0.6762), 'f1_std': np.float64(0.009)}
XGBoost | all columns : {'f1': np.float64(0.6278), 'roc_auc': np.float64(0.864), 'pr_auc': np.float64(0.7364), 'f1_std': np.float64(0.0744)}
XGBoost | no Agent_te : {'f1': np.float64(0.6067), 'roc_auc': np.float64(0.8403), 'pr_auc': np.float64(0.7048), 'f1_std': np.float64(0.0074)}


In [ ]:
# selected = ["RandomForest", "XGBoost", "MLP"]
# baseline_selected = baseline[baseline["model"].isin(selected)]
# display(baseline_selected)